# DataHack 2026: Initial Data Analysis

## The 500-Meter Problem

**Objective:** Analyze accessibility of recycling infrastructure for Hong Kong's public housing residents

### Data Available:
- **8,858** recycling collection points across Hong Kong
- **241** public housing estates
- **~806,600** rental flats
- **~2.2 million** residents (29% of HK population)

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from math import radians, cos, sin, asin, sqrt

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

%matplotlib inline
sns.set_style('whitegrid')

## Step 1: Load and Clean Data

In [ ]:
# Load collection points
df_collection = pd.read_csv('../data/raw/collection_points.csv', encoding='utf-8-sig')

print(f"Collection Points: {len(df_collection):,} locations")
print(f"Columns: {list(df_collection.columns)}")
df_collection.head()

In [ ]:
# Load public housing estates
import json

with open('../data/raw/public_housing.json') as f:
    ph_data = json.load(f)

# Convert to DataFrame
estates = []
for e in ph_data:
    # Parse flat count
    flats_str = e['No. of Rental Flats']['en']
    try:
        num_flats = int(flats_str.split('as at')[0].strip().replace(' ', '').replace(',', ''))
    except:
        num_flats = None
    
    estates.append({
        'name': e['Estate Name']['en'],
        'district': e['District Name']['en'],
        'region': e['Region Name']['en'],
        'latitude': e['Estate Map Latitude'],
        'longitude': e['Estate Map Longitude'],
        'num_flats': num_flats,
        'est_population': int(num_flats * 2.7) if num_flats else None
    })

df_estates = pd.DataFrame(estates)

print(f"Public Housing Estates: {len(df_estates):,}")
print(f"Total flats: {df_estates['num_flats'].sum():,}")
print(f"Est. population: {df_estates['est_population'].sum():,}")
df_estates.head()

## Step 2: Create Spatial GeoDataFrames

In [ ]:
# Create GeoDataFrame for collection points
geometry_cp = [Point(xy) for xy in zip(df_collection['lgt'], df_collection['lat'])]
gdf_collection = gpd.GeoDataFrame(df_collection, geometry=geometry_cp, crs='EPSG:4326')

print(f"Collection points GeoDataFrame: {len(gdf_collection):,} points")
print(f"Bounds: {gdf_collection.total_bounds}")

In [ ]:
# Create GeoDataFrame for estates
geometry_estates = [Point(xy) for xy in zip(df_estates['longitude'], df_estates['latitude'])]
gdf_estates = gpd.GeoDataFrame(df_estates, geometry=geometry_estates, crs='EPSG:4326')

print(f"Estates GeoDataFrame: {len(gdf_estates):,} estates")
print(f"Bounds: {gdf_estates.total_bounds}")

## Step 3: Distance Calculation Function

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees)
    Returns distance in meters
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    
    # Radius of earth in meters
    r = 6371000
    
    return c * r

# Test the function
test_dist = haversine_distance(22.330105, 114.2031, 22.335, 114.21)
print(f"Test distance: {test_dist:.1f} meters")

## Step 4: Calculate Nearest Collection Point for Each Estate

In [ ]:
# For each estate, find the nearest collection point
def find_nearest_distance(estate_row, collection_gdf):
    """Find distance to nearest collection point"""
    estate_lat = estate_row['latitude']
    estate_lon = estate_row['longitude']
    
    # Calculate distance to all collection points
    distances = collection_gdf.apply(
        lambda cp: haversine_distance(estate_lat, estate_lon, cp['lat'], cp['lgt']),
        axis=1
    )
    
    return distances.min()

print("Calculating distances (this may take a minute)...")
gdf_estates['distance_to_nearest'] = gdf_estates.apply(
    lambda row: find_nearest_distance(row, gdf_collection), 
    axis=1
)

print("Done!")
gdf_estates.head()

## Step 5: Categorize Accessibility

In [ ]:
# Categorize estates by accessibility
def categorize_access(distance):
    if distance < 300:
        return 'Well-served (<300m)'
    elif distance < 500:
        return 'Moderate (300-500m)'
    else:
        return 'Underserved (>500m)'

gdf_estates['accessibility'] = gdf_estates['distance_to_nearest'].apply(categorize_access)

# Summary statistics
print("\nACCESSIBILITY SUMMARY")
print("=" * 70)
print("\nBy estate count:")
print(gdf_estates['accessibility'].value_counts())

print("\nBy population:")
pop_by_access = gdf_estates.groupby('accessibility')['est_population'].sum()
print(pop_by_access)

total_pop = gdf_estates['est_population'].sum()
underserved_pop = gdf_estates[gdf_estates['accessibility'] == 'Underserved (>500m)']['est_population'].sum()

print(f"\n% of public housing residents underserved: {underserved_pop / total_pop * 100:.1f}%")
print(f"Underserved population: {underserved_pop:,} people")

## Step 6: Visualize Results

In [ ]:
# Distribution of distances
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(gdf_estates['distance_to_nearest'], bins=50, edgecolor='black')
axes[0].axvline(x=300, color='green', linestyle='--', label='300m threshold')
axes[0].axvline(x=500, color='red', linestyle='--', label='500m threshold')
axes[0].set_xlabel('Distance to Nearest Collection Point (meters)')
axes[0].set_ylabel('Number of Estates')
axes[0].set_title('Distribution of Distances')
axes[0].legend()

# Box plot by region
gdf_estates.boxplot(column='distance_to_nearest', by='region', ax=axes[1])
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Distance (meters)')
axes[1].set_title('Distance by Region')

plt.tight_layout()
plt.savefig('../visualizations/distance_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart of accessibility categories
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# By count
access_counts = gdf_estates['accessibility'].value_counts()
colors = ['green', 'yellow', 'red']
access_counts.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Estates by Accessibility Category')
axes[0].set_ylabel('Number of Estates')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

# By population
pop_by_access.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Population by Accessibility Category')
axes[1].set_ylabel('Population')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../visualizations/accessibility_categories.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 7: Interactive Map

In [ ]:
# Create interactive map
m = folium.Map(location=[22.35, 114.15], zoom_start=11)

# Add collection points (small blue dots)
for idx, row in gdf_collection.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lgt']],
        radius=1,
        color='blue',
        fill=True,
        opacity=0.3
    ).add_to(m)

# Add estates (colored by accessibility)
color_map = {
    'Well-served (<300m)': 'green',
    'Moderate (300-500m)': 'orange',
    'Underserved (>500m)': 'red'
}

for idx, row in gdf_estates.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color=color_map[row['accessibility']],
        fill=True,
        popup=f"{row['name']}<br>Distance: {row['distance_to_nearest']:.0f}m<br>Pop: {row['est_population']:,}"
    ).add_to(m)

# Save map
m.save('../visualizations/accessibility_map_current.html')
print("Map saved to: visualizations/accessibility_map_current.html")
m

## Step 8: Identify Priority Underserved Estates

In [ ]:
# Get underserved estates sorted by population
underserved = gdf_estates[gdf_estates['accessibility'] == 'Underserved (>500m)'].copy()
underserved = underserved.sort_values('est_population', ascending=False)

print(f"\nTOP 20 PRIORITY ESTATES (Underserved, Highest Population)")
print("=" * 70)
print(underserved[['name', 'district', 'distance_to_nearest', 'est_population']].head(20).to_string())

# Save to CSV
underserved.to_csv('../data/processed/underserved_estates.csv', index=False)
print(f"\nSaved {len(underserved)} underserved estates to data/processed/underserved_estates.csv")

## Step 9: Save Processed Data

In [ ]:
# Save all estates with analysis
gdf_estates.to_file('../data/processed/estates_with_accessibility.geojson', driver='GeoJSON')
gdf_collection.to_file('../data/processed/collection_points.geojson', driver='GeoJSON')

print("Saved processed data:")
print("  - estates_with_accessibility.geojson")
print("  - collection_points.geojson")

## Summary

### Key Findings:
1. Analyzed **241 public housing estates** covering **~2.2M residents**
2. Mapped **8,858 recycling collection points** across Hong Kong
3. Identified estates >500m from nearest collection point
4. Calculated population impact of accessibility gaps

### Next Steps:
1. Run greedy optimization algorithm to place new micro-hubs
2. Calculate before/after impact metrics
3. Create final visualizations
4. Build presentation